# Building Your First Audiobook with vorpal

This notebook walks you through converting a book file into a chaptered M4B audiobook using vorpal. By the end you'll have a `.m4b` file you can load directly into Apple Books, Smart AudioBook Player, or any audiobook-capable app — complete with chapter markers and cover art.

We'll use a short EPUB from Project Gutenberg as the example input, but the same steps apply to PDF and plain-text files. The full pipeline runs locally on your machine: no cloud calls, no accounts required.

## Prerequisites

Before running this notebook, make sure you have the following:

- **vorpal installed** in your Python environment (`pip install -e .` from the repo root, or use the pre-built `vorpal-box` Docker container)
- **ffmpeg** on your PATH (Windows: `C:\ffmpeg\bin\`; Linux: `apt-get install ffmpeg`)
- **Tesseract** on your PATH — only needed for PDF input with scanned pages (Windows: `C:\Program Files\Tesseract-OCR\`; Linux: `apt-get install tesseract-ocr`)
- **GPU recommended but not required** — synthesis on GPU takes ~25 minutes for a full book; CPU takes hours. For this notebook we use `--end-page 30` to keep things fast on any hardware.

If you're running inside the `vorpal-box` Docker container, everything above is already set up — just start from Cell 3.

In [ ]:
import subprocess
import json
import pathlib

# Quick sanity check: can we import vorpal and see the voice registry?
from vorpal.tts.voices import VOICE_REGISTRY
print(f"{len(VOICE_REGISTRY)} voices available")
print("Voices:", list(VOICE_REGISTRY.keys()))

## Choosing Your Input

vorpal accepts three input formats:

**EPUB** is the easiest path. EPUBs have clean internal structure — the file already knows its chapters, metadata (title, author, cover art), and the text is already machine-readable. No OCR needed, chapter detection is very reliable, and builds are fast.

**PDF** works well for born-digital PDFs (the text layer is present). Scanned PDFs go through Tesseract OCR first, which adds time and can introduce occasional recognition errors. Two-page-spread scans and low-contrast pages are handled but may need review.

**TXT** is the simplest format but has no metadata. You'll want to pass `--title` and `--author` flags so the M4B tags are correct. Chapter detection falls back to heuristics (lines that look like headings).

For your first build, **we recommend downloading a small EPUB from Project Gutenberg**. Their collection covers tens of thousands of public-domain books. Good starter choices: *The Time Machine* by H.G. Wells (short, clean), *Flatland* by Edwin Abbott (short, mathematical), or *The Adventures of Sherlock Holmes* (well-structured, familiar).

The download cell below shows the pattern — swap in any Gutenberg EPUB ID you like.

In [ ]:
# Download a small public-domain EPUB from Project Gutenberg.
# Replace with any .epub you have locally — just update `epub_path` below.
#
# Gutenberg EPUB URL pattern:
#   https://www.gutenberg.org/ebooks/<ID>.epub.images
# or the no-images variant:
#   https://www.gutenberg.org/ebooks/<ID>.epub.noimages
#
# Examples:
#   35 = The Time Machine (H.G. Wells)
#   97 = The War of the Worlds (H.G. Wells)
#  201 = Flatland (Edwin Abbott)
# 1661 = The Adventures of Sherlock Holmes (Doyle)

GUTENBERG_ID = 35  # The Time Machine — 91 pages, ~8 chapters, good test size
epub_url = f"https://www.gutenberg.org/ebooks/{GUTENBERG_ID}.epub.noimages"
epub_path = pathlib.Path("book.epub")

if not epub_path.exists():
    import urllib.request
    print(f"Downloading {epub_url} ...")
    urllib.request.urlretrieve(epub_url, epub_path)
    print(f"Saved to {epub_path} ({epub_path.stat().st_size / 1024:.0f} KB)")
else:
    print(f"Already have {epub_path} ({epub_path.stat().st_size / 1024:.0f} KB) — skipping download")

## Your First Build

`vorpal build` runs the full pipeline from input file to finished M4B. The stages, in order:

1. **Extract** — reads the file, produces raw text with page/block geometry
2. **Segment** — detects chapter boundaries using the EPUB outline, printed TOC, or heuristics
3. **Normalize** — expands abbreviations and numbers, splits text into TTS-safe chunks at natural sentence and paragraph boundaries
4. **Synthesize** — converts each chunk to audio using the Kokoro TTS model (cached per text+voice+speed)
5. **Master** — applies EBU R128 loudness normalization, assembles chapters into a single M4B with markers

Key flags for this first run:

- `--output my_first_book` sets the stem for the output file and workdir
- `--end-page 30` stops after page 30 — keeps the first run fast on any hardware

Remove `--end-page` when you want to build the full book.

In [ ]:
result = subprocess.run(
    ["vorpal", "build", "book.epub", "--output", "my_first_book", "--end-page", "30"],
    capture_output=True,
    text=True
)

# Print the last 2000 characters of stdout (progress log), or stderr on failure
if result.stdout:
    print(result.stdout[-2000:])
else:
    print(result.stderr[-1000:])

print(f"\nReturn code: {result.returncode}")

## What Was Produced?

After a successful build you'll find two things:

**`my_first_book.m4b`** — the finished audiobook file. This is what you copy to your phone or import into your audio player.

**`my_first_book_workdir/`** — the pipeline's working directory. Everything intermediate lives here:

- `book.json` — the manifest: a single JSON file that records every decision the pipeline made (source metadata, per-chapter settings, synthesis parameters). Every stage reads and writes this file. It's the single source of truth.
- `pages.jsonl` — the extracted text, one JSON object per page/block
- `chapters/` — per-chapter WAV files before mastering
- `chunks/` — the TTS cache: one WAV per synthesized text chunk, named by content hash. Re-running the build skips any chunk that's already been synthesized.
- `report.md` — a build summary with timing, chapter list, and word counts

The workdir is safe to delete once you're happy with the M4B — but keeping it means future rebuilds (different voice, different settings) only re-synthesize what changed.

In [ ]:
wd = pathlib.Path("my_first_book_workdir")

if wd.exists():
    print("Workdir contents:")
    for item in sorted(wd.iterdir()):
        if item.is_dir():
            child_count = len(list(item.iterdir()))
            print(f"  {item.name}/  ({child_count} items)")
        else:
            print(f"  {item.name}  ({item.stat().st_size / 1024:.0f} KB)")

    manifest_path = wd / "book.json"
    if manifest_path.exists():
        book = json.loads(manifest_path.read_text(encoding="utf-8"))
        print(f"\nTitle:    {book['source']['title']}")
        print(f"Author:   {book['source'].get('author', 'unknown')}")
        included = [c for c in book.get('chapters', []) if c.get('include', True)]
        print(f"Chapters: {len(included)} included")
else:
    print("Workdir not found — run the build cell above first.")

## Reviewing Chapters

`vorpal review` is a lighter command that shows you how the book was segmented without rebuilding the audio. It prints each detected chapter with its title, word count, source (was the boundary found in the EPUB outline, a printed TOC, or by heuristic?), and a confidence score.

Chapters flagged with low confidence or marked `include: false` (front matter, bibliography, index) are highlighted for your attention. You can edit `book.json` directly to:

- Change a chapter title (affects the chapter marker in the M4B)
- Set `"include": false` to exclude a chapter from synthesis
- Set `"spoken_intro"` to inject a spoken line before the chapter audio (e.g. `"Chapter One."`)

After editing, run `vorpal review --approve` to lock the manifest and tell the pipeline the chapter structure is correct. The next `vorpal build` will pick up your changes.

For books where chapter detection is very reliable (most clean EPUBs), you can skip the review step entirely — `vorpal build` will proceed through all stages automatically.

In [ ]:
review = subprocess.run(
    ["vorpal", "review", "book.epub"],
    capture_output=True,
    text=True
)
print(review.stdout if review.stdout else review.stderr)

## The Output File

The `.m4b` file is an MPEG-4 audio container — the standard format for audiobooks on Apple platforms and widely supported everywhere else.

What's inside:

- **AAC audio** — the narration, loudness-normalized to EBU R128 (-18 LUFS integrated, -1 dBTP peak)
- **Chapter markers** — each chapter is a named bookmark; your audio player's chapter list comes from these
- **Cover art** — extracted from the EPUB or PDF if available, or left blank
- **ID3-style metadata** — title, author, narrator (set to the voice name)

vorpal also produces an `MP3` side product in the workdir if you need broad compatibility (some older devices don't support M4B).

**Getting it onto your phone:**

- **iPhone / Apple Books**: drag the `.m4b` into your iTunes/Finder library, sync, or AirDrop it directly. Apple Books will recognize it as an audiobook and show chapter marks.
- **Android / Smart AudioBook Player**: copy the file to your phone's storage (USB, cloud sync, whatever you prefer). Smart AudioBook Player picks up chapter markers automatically.
- **Desktop**: VLC, foobar2000, and most media players handle M4B. Chapter markers appear in the chapter/track list.

In [ ]:
m4b = pathlib.Path("my_first_book.m4b")

if m4b.exists():
    size_mb = m4b.stat().st_size / 1e6
    print(f"{m4b.name}: {size_mb:.1f} MB")

    # Try to get duration via ffprobe if available
    probe = subprocess.run(
        ["ffprobe", "-v", "quiet", "-print_format", "json", "-show_format", str(m4b)],
        capture_output=True, text=True
    )
    if probe.returncode == 0:
        info = json.loads(probe.stdout)
        duration_s = float(info["format"]["duration"])
        h, rem = divmod(int(duration_s), 3600)
        m, s = divmod(rem, 60)
        print(f"Duration:  {h}h {m:02d}m {s:02d}s")
    else:
        print("(ffprobe not available — cannot read duration)")
else:
    print("Build not yet complete — run the build cell above first.")

## Next Steps

You've built your first audiobook. Here's where to go next:

**Try a different voice** — the default is `af_heart` (warm American female). To use a different narrator, add `--voice blend_deep_steady` (or any other voice name) to the build command. See `notebooks/02_voices.ipynb` for a full tour of the voice suite and guidance on matching voices to book genres.

**Understand the pipeline internals** — `notebooks/03_manifest_and_pipeline.ipynb` explains the seven pipeline stages, how `book.json` works, what the chunk cache is doing, and how to inspect a build without re-running it.

**Build a full book** — remove `--end-page 30` from the build command. On a GPU this takes ~25 minutes for a typical novel; on CPU it may take several hours. The pipeline is fully resumable — if it stops mid-way, re-running `vorpal build` picks up from the last completed chunk.

**Try a scanned PDF** — download a public-domain scan from the Internet Archive (`archive.org`) and build it. The OCR + chapter detection path is more interesting than clean EPUB, and the review step becomes more useful.